In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.metrics import silhouette_score, davies_bouldin_score, adjusted_rand_score, normalized_mutual_info_score
import warnings
warnings.filterwarnings('ignore')

# 1. Load Data

In [ ]:
df = pd.read_csv("Iris.csv")
print(df.info())

x = df.drop(['Id', 'Species'], axis=1)
y = df.Species

# 2. Data Standardization

In [ ]:
std = StandardScaler()
x_new = std.fit_transform(x)
std_df = pd.DataFrame(x_new, columns=x.columns)

# 3. K-Means Elbow Method (Silhouette Score)

In [ ]:
k_values = [i for i in range(2, 10)]
results = {}

for k in k_values:
    kmeans_model = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans_model.fit_predict(std_df)
    score = silhouette_score(x, labels)
    results[k] = score
    print(f"K={k} -> Silhouette Score: {score}")

best_k = max(results, key=results.get)
print(f"\nBest performing K based on Silhouette Score: {best_k}")

# Plot Elbow Method

In [ ]:
plt.plot(k_values, results.values())
plt.xlabel("Cluster")
plt.ylabel("silhouette score")
plt.title("Elbow Method")
plt.show()

# 4. K-Means Clustering

In [ ]:
km_model = KMeans(n_clusters=3)
y_pred = km_model.fit_predict(std_df)

sns.scatterplot(x=x.SepalLengthCm, y=x.PetalLengthCm, hue=y_pred, palette='rainbow')
plt.show()

# 5. Hierarchical Clustering

In [ ]:
plt.figure(figsize=(8, 4))
Z = linkage(x_new, method='ward')
dendrogram(Z, truncate_mode='lastp', p=30, leaf_font_size=8)
plt.title("Hierarchical Clustering Dendrogram")
plt.xlabel("Clusters")
plt.ylabel("Distance")
plt.show()

# 6. Agglomerative Clustering

In [ ]:
n_clusters = 3
model = AgglomerativeClustering(n_clusters=n_clusters, linkage='ward')
labels = model.fit_predict(x_new)
print("\nCluster labels:")
print(labels)

plt.figure(figsize=(8, 6))
plt.scatter(x_new[:, 0], x_new[:, 1], c=labels, cmap='rainbow', s=50)
plt.xlabel("Sepal Length (standardized)")
plt.ylabel("Sepal Width (standardized)")
plt.title("Agglomerative Clustering")
plt.show()

# 7. Cluster Evaluation

In [ ]:
silhouette = silhouette_score(x_new, labels)
db_score = davies_bouldin_score(x_new, labels)
print("The Internal Evaluation is: ")
print("Silhouette Score:", silhouette)
print("Davies-Bouldin Score:", db_score)

ari = adjusted_rand_score(y, labels)
nmi = normalized_mutual_info_score(y, labels)
print("The External Evaluation is:-")
print("Adjusted Rand Index is", ari)
print("Normalized Mutual Information is:", nmi)

# 8. Gaussian Mixture Model (GMM)

In [ ]:
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture

pca = PCA(n_components=2)
x_pca = pca.fit_transform(x_new)

gmm = GaussianMixture(n_components=3, covariance_type='full', random_state=42)
labels = gmm.fit_predict(x_pca)
print("\nCluster labels:")
print(labels)

probabilities = gmm.predict_proba(x_pca)
print("First 5 probability vectors:")
print(probabilities[:5])

# 9. GMM Decision Boundaries

In [ ]:
x_min = x_pca[:, 0].min() - 1
x_max = x_pca[:, 0].max() + 1

y_min = x_pca[:, 1].min() - 1
y_max = x_pca[:, 1].max() + 1

xx, yy = np.meshgrid(
    np.linspace(x_min, x_max, 500),
    np.linspace(y_min, y_max, 500)
)

grid = np.c_[xx.ravel(), yy.ravel()]
Z = gmm.predict(grid)
Z = Z.reshape(xx.shape)

plt.figure(figsize=(10, 7))
plt.contourf(xx, yy, Z, alpha=0.2, cmap='viridis')
plt.scatter(x_pca[:, 0], x_pca[:, 1], c=labels, cmap='rainbow', edgecolors='black', s=50)
plt.xlabel("PC-1")
plt.ylabel("PC-2")
plt.title("GMM Decision Boundaries")
plt.show()